# Postprocess into rasters and vector datasets




In [1]:
import os
import geopandas as gpd
import geoutils as gu
import numpy as np
import xdem
from osgeo import gdal, ogr, osr
import rasterstats
from osgeo_utils import gdal_calc


import subkart

In [2]:
nodata = 255
crs = "EPSG:25833"

In [3]:
classifier = subkart.utils.load_classifier()

## Post processing

In [4]:
predict_file_unmapped = "predict_unmapped.tif"
prob_file = "3band_probability.tif"

predict_file_remapped = "predict_remapped.tif"


In [5]:
fname = subkart.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}", "norge", "latest", crs.split(":")[1]
)
predict_file = f"{fname}.tif"

fname_prob = subkart.utils.to_filename(
    f"nisjedata-substrat-{classifier.__class__.__name__.lower()}-probability", "norge", "latest", crs.split(":")[1]
)
prob_file_processed = f"{fname_prob}.tif"

In [6]:
gdal.UseExceptions()

prediction_files = [f"{r}_prediction.tif" for r in subkart.sources.REGIONS]
probability_files = [f"{r}_probability.tif" for r in subkart.sources.REGIONS]

subkart.utils.merge_rasters(prediction_files, predict_file_unmapped, nodata=nodata)
subkart.utils.merge_rasters(probability_files, prob_file, nodata=-9999)


# Create processed prediction raster:

Remap class 1 (blanding) → highest-probability class (0=løsbunn or 2=fastbunn) using gdal_calc

In [7]:
subkart.utils.remap_prediction(predict_file_unmapped, prob_file, predict_file_remapped, nodata=nodata)

## Create processed 1-band probability raster from the 3-band source

* class 0 (løsbunn)        band 1 = P(class=0)
* class 2 (fastbunn)       band 3 = P(class=2)

In [8]:
subkart.utils.create_probability_raster(predict_file_remapped, prob_file, prob_file_processed, nodata=nodata)

## Filter isolated low-probability pixels

Use `gdal.SieveFilter` (threshold=1, 4-connected) to replace single isolated pixels with the
surrounding class, then update `prob_file_processed` for changed pixels.

In [9]:
PROB_THRESHOLD = 0.60  # Filter probability threshold for filtering noise pixels

In [10]:
subkart.utils.filter_isolated_pixels(
    predict_file_remapped, prob_file_processed, predict_file, nodata=nodata, prob_threshold=PROB_THRESHOLD
)

## Vectorize processed prediction raster

In [11]:
subkart.vectorize.with_gdal(
    predict_file, "polygons_processed.gpkg", epsg_code=int(crs.split(":")[1])
)

gdf = gpd.read_file("polygons_processed.gpkg").explode()

reverse_map = {v: k for k, v in subkart.labelling.BUNNTYPE_MAPPING.items()}
gdf["BunnType"] = gdf["DN"].map(reverse_map)

# Compute mean probability per polygon from the processed 1-band probability raster
stats = rasterstats.zonal_stats(
    gdf,
    prob_file_processed,
    stats=["mean"],
    nodata=-9999,
)
gdf["Sannsynlighet"] = [s["mean"]*100 for s in stats]

gdf.to_file(f"{fname}.gpkg", driver="GPKG", layer="bunntyper")
gdf.to_parquet(f"{fname}.geo.parquet", compression="snappy")


Polygons saved to polygons_processed.gpkg


In [12]:
subkart.utils.to_postgis(gdf, fname)

Table nisjedata_substrat_xgbclassifier_norge_latest uploaded to PostGIS.
